In [1]:
import copy
import json
import math
from collections import defaultdict
from datetime import datetime, timedelta
from statistics import median
from typing import Dict, List, Any, Optional, Tuple, NamedTuple
from pathlib import Path


def parse_time(time_str: str) -> datetime:
    """Parse ISO timestamp without altering timezone information."""
    try:
        return datetime.fromisoformat(time_str)
    except ValueError:
        return datetime.strptime(time_str, "%Y-%m-%dT%H:%M:%S")


def normalize_date(date_str: Optional[str]) -> Optional[str]:
    """
    Normalize a date string to YYYY-MM-DD format.
    
    Handles various date formats and returns None if parsing fails.
    
    Args:
        date_str: Date string in various formats (YYYY-MM-DD, YYYY/MM/DD, etc.)
        
    Returns:
        Normalized date string in YYYY-MM-DD format, or None if parsing fails
    """
    if not date_str:
        return None
    
    # Try common date formats
    date_formats = [
        "%Y-%m-%d",      # ISO format
        "%Y/%m/%d",      # Slash format
        "%Y.%m.%d",      # Dot format
        "%d-%m-%Y",      # European format
        "%d/%m/%Y",      # European slash format
        "%m/%d/%Y",      # US format
        "%Y-%m-%dT%H:%M:%S",  # ISO with time
        "%Y-%m-%d %H:%M:%S",  # Space-separated
    ]
    
    # First try parsing as-is
    for fmt in date_formats:
        try:
            dt = datetime.strptime(str(date_str).strip(), fmt)
            return dt.date().isoformat()  # Return as YYYY-MM-DD
        except (ValueError, TypeError):
            continue
    
    # Try parsing as ISO format (handles partial dates)
    try:
        dt = datetime.fromisoformat(str(date_str).strip())
        return dt.date().isoformat()
    except (ValueError, TypeError):
        pass
    
    # If all parsing fails, return None
    return None


def get_participant_id(
    event: Dict[str, Any],
    object_map: Dict[str, Dict[str, Any]]
) -> Optional[str]:
    """Extract participant ID from event relationships."""
    for rel in event.get('relationships', []):
        if rel.get('qualifier') != 'corr':
            continue

        rel_obj_id = rel.get('objectId') or rel.get('id')
        if not rel_obj_id:
            continue

        obj = object_map.get(rel_obj_id)
        if obj and obj.get('type') == 'Participant':
            return rel_obj_id
    return None


def add_relationship(
    relationships: List[Dict[str, Any]],
    target_id: str,
    qualifier: str
) -> None:
    """Add a relationship entry if it doesn't already exist."""
    for rel in relationships:
        existing_id = rel.get('objectId') or rel.get('id')
        if existing_id != target_id:
            continue
        if rel.get('qualifier') != qualifier:
            continue
        return

    relationships.append({
        'objectId': target_id,
        'qualifier': qualifier
    })


def create_bidirectional_relationship(
    obj1: Dict[str, Any],
    obj2: Dict[str, Any],
    qualifier1: str,
    qualifier2: str
) -> None:
    """
    Create bidirectional relationships between two objects/events.
    
    Args:
        obj1: First object/event (gets relationship with qualifier1)
        obj2: Second object/event (gets relationship with qualifier2)
        qualifier1: Qualifier for obj1 -> obj2 relationship
        qualifier2: Qualifier for obj2 -> obj1 relationship
    """
    if not obj1.get('relationships'):
        obj1['relationships'] = []
    if not obj2.get('relationships'):
        obj2['relationships'] = []
    
    add_relationship(obj1['relationships'], obj2['id'], qualifier1)
    add_relationship(obj2['relationships'], obj1['id'], qualifier2)


def link_inference_relationship(
    source: Dict[str, Any],
    target: Dict[str, Any]
) -> None:
    """
    Create bidirectional inference relationship: source infers target.
    
    Creates:
    - source -> target: 'infer' (source infers target)
    - target -> source: 'inferredBy' (target is inferred by source)
    """
    create_bidirectional_relationship(source, target, 'infer', 'inferredBy')


def link_correlation_relationship(
    obj1: Dict[str, Any],
    obj2: Dict[str, Any]
) -> None:
    """
    Create bidirectional correlation relationship.
    
    Creates:
    - obj1 -> obj2: 'corr'
    - obj2 -> obj1: 'corr'
    """
    create_bidirectional_relationship(obj1, obj2, 'corr', 'corr')


def register_behavior_event_type(
    ocel_data: Dict[str, Any],
    event_type_name: str,
    attributes: List[Dict[str, Any]],
    verbose: bool = False
) -> bool:
    """
    Register a behavior event type in OCEL data if it doesn't already exist.
    
    Args:
        ocel_data: OCEL data dictionary
        event_type_name: Name of the behavior event type
        attributes: List of attribute definitions
        verbose: Whether to print messages
        
    Returns:
        True if behavior event type was added, False if it already existed
    """
    behavior_event_types = ocel_data.get('behaviorEventTypes', [])
    
    if any(et['name'] == event_type_name for et in behavior_event_types):
        return False
    
    behavior_event_types.append({
        'name': event_type_name,
        'attributes': attributes
    })
    
    if verbose:
        print(f"  Added {event_type_name} behavior event type")
    
    ocel_data['behaviorEventTypes'] = behavior_event_types
    return True


def register_object_type(
    ocel_data: Dict[str, Any],
    object_type_name: str,
    attributes: List[Dict[str, Any]],
    verbose: bool = False
) -> bool:
    """
    Register an object type in OCEL data if it doesn't already exist.
    
    Args:
        ocel_data: OCEL data dictionary
        object_type_name: Name of the object type
        attributes: List of attribute definitions
        verbose: Whether to print messages
        
    Returns:
        True if object type was added, False if it already existed
    """
    object_types = ocel_data.get('objectTypes', [])
    
    if any(ot['name'] == object_type_name for ot in object_types):
        return False
    
    object_types.append({
        'name': object_type_name,
        'attributes': attributes
    })
    
    if verbose:
        print(f"  Added {object_type_name} object type")
    
    ocel_data['objectTypes'] = object_types
    return True


def get_glucose_value(event: Dict[str, Any]) -> Optional[float]:
    """Extract glucose value from sensor event, handling non-numeric values."""
    for attr in event.get('sensorEventTypeAttributes', []):
        if attr['name'] == 'value':
            value = attr['value']
            if isinstance(value, str):
                value_lower = value.lower()
                if value_lower in ['low', 'high', 'none', 'na', '']:
                    return None
            try:
                return float(value)
            except (ValueError, TypeError):
                return None
    return None


def round_for_storage(
    value: Optional[float],
    ndigits: int = 2
) -> Optional[float]:
    """Round numeric attribute values for storage."""
    if value is None:
        return None
    return round(value, ndigits)


class IndexedEvent(NamedTuple):
    """Cached event data for efficient lookups (generic)."""
    event: Dict[str, Any]
    event_type: str
    parsed_time: Optional[datetime]  # For single-timestamp events (glucose, meal, bolus, sleep)
    start_time: Optional[datetime]  # For range events (activity)
    end_time: Optional[datetime]  # For range events (activity)
    glucose_value: Optional[float]  # For glucose events
    start_time_s: Optional[float]  # For activity events
    duration_s: Optional[float]  # For activity events
    participant_id: str
    calendar_day: Optional[str]  # For sleep, activity events


class IndexedObject(NamedTuple):
    """Cached object data for efficient lookups (generic)."""
    object: Dict[str, Any]
    object_type: str
    participant_id: Optional[str]
    calendar_day: Optional[str]


class InferenceConfig:
    """Configuration constants for inference pipelines."""
    # Glucose thresholds (mmol/L)
    LOWER_THRESHOLD = 3.9
    UPPER_THRESHOLD = 10.0
    
    # Postprandial window parameters
    PRE_MEAL_WINDOW_MINUTES = 60
    POST_MEAL_WINDOW_MINUTES = 180
    PEAK_WINDOW_MINUTES = 30
    RECOVERY_WINDOW_MINUTES = 60


class OCELData:
    """Simple wrapper for OCEL data structure to enable dependency injection."""
    
    def __init__(self, data: Dict[str, Any]):
        self.data = data
        self._sensor_events = data.get('sensorEvents', [])
        self._behavior_events = data.get('behaviorEvents', [])
        self._objects = data.get('objects', [])
        self._object_map = {obj['id']: obj for obj in self._objects}
    
    @property
    def sensor_events(self) -> List[Dict[str, Any]]:
        return self._sensor_events
    
    @property
    def behavior_events(self) -> List[Dict[str, Any]]:
        return self._behavior_events
    
    @property
    def objects(self) -> List[Dict[str, Any]]:
        return self._objects
    
    @property
    def object_map(self) -> Dict[str, Dict[str, Any]]:
        return self._object_map
    
    def get(self, key: str, default: Any = None) -> Any:
        return self.data.get(key, default)
    
    def to_dict(self) -> Dict[str, Any]:
        return self.data.copy()


class BaseInference:
    """Base class for inference pipelines with shared functionality."""
    
    def __init__(self, ocel_data: Dict[str, Any]):
        """
        Initialize with OCEL data.
        
        Args:
            ocel_data: Dictionary containing Sensor-Augmented OCEL data
        """
        self.ocel_data = ocel_data
        self.data = OCELData(ocel_data)
        self.sensor_events = self.data.sensor_events
        self.behavior_events = self.data.behavior_events
        self.objects = self.data.objects
        self.object_map = self.data.object_map
        
        # Common state
        self.behavior_event_counter = defaultdict(int)
        self.inferred_objects: List[Dict[str, Any]] = []
        self.inferred_behavior_events: List[Dict[str, Any]] = []
        self._objects_by_type: Dict[str, List[IndexedObject]] = {}
        self._participants: List[IndexedObject] = []
        
        # Mutation protection (standardized across all classes)
        self._event_copies: Dict[str, Dict[str, Any]] = {}
        self._object_copies: Dict[str, Dict[str, Any]] = {}
    
    def _build_object_index(self) -> None:
        """Build indexed structure for objects by type, with cached participant IDs."""
        self._objects_by_type = {}
        self._participants = []
        
        for obj in self.objects:
            obj_type = obj.get('type', 'Unknown')
            obj_id = obj.get('id')
            
            # Extract participant_id from relationships if present
            participant_id = None
            for rel in obj.get('relationships', []):
                rel_target = rel.get('objectId') or rel.get('id')
                if not rel_target:
                    continue
                if rel.get('qualifier') != 'corr':
                    continue
                target_obj = self.object_map.get(rel_target)
                if target_obj and target_obj.get('type') == 'Participant':
                    participant_id = rel_target
                    break
            
            # Extract calendar_day from attributes if present
            calendar_day = None
            for attr in obj.get('attributes', []):
                if attr.get('name') == 'calendar_day':
                    raw_calendar_day = attr.get('value')
                    # Normalize date format to ensure consistent matching
                    calendar_day = normalize_date(raw_calendar_day)
                    if calendar_day:
                        break
            
            # Create indexed object
            indexed_obj = IndexedObject(
                object=obj,
                object_type=obj_type,
                participant_id=participant_id,
                calendar_day=calendar_day
            )
            
            # Index by type
            if obj_type not in self._objects_by_type:
                self._objects_by_type[obj_type] = []
            self._objects_by_type[obj_type].append(indexed_obj)
            
            # Special handling for Participant objects
            if obj_type == 'Participant':
                # For Participant objects, participant_id is the object's ID
                self._participants.append(IndexedObject(
                    object=obj,
                    object_type=obj_type,
                    participant_id=obj_id,
                    calendar_day=None
                ))
    
    def _build_simple_event_index(
        self,
        events: List[Dict[str, Any]],
        event_type_name: str,
        event_type_field: str,
        event_type_value: str,
        index_dict: Dict,
        use_calendar_day: bool = False,
        extract_glucose_value: bool = False
    ) -> None:
        """
        Build indexed structure for events by participant (and optionally by calendar_day).
        
        Args:
            events: List of events to index
            event_type_name: Event type name for IndexedEvent ('BloodGlucose', 'Meal', etc.)
            event_type_field: Field to check ('sensorEventType' or 'behaviorEventType')
            event_type_value: Value to filter by ('BloodGlucose', 'Meal', etc.)
            index_dict: Dictionary to store indexed events (by participant or by (participant, day))
            use_calendar_day: Whether to index by participant + calendar_day
            extract_glucose_value: Whether to extract glucose_value for glucose events
        """
        for event in events:
            if event.get(event_type_field) != event_type_value:
                continue
            
            participant_id = get_participant_id(event, self.object_map)
            if not participant_id:
                continue
            
            try:
                parsed_time = parse_time(event['time'])
            except (ValueError, AttributeError, TypeError):
                continue
            
            glucose_value = None
            if extract_glucose_value:
                glucose_value = get_glucose_value(event)
                if glucose_value is None:
                    continue
            
            calendar_day = None
            if use_calendar_day:
                # Check for calendar_day (used by some events like DGV objects) or calendar_date (used by SleepTime events)
                for attr in event.get('behaviorEventTypeAttributes', []):
                    attr_name = attr.get('name')
                    if attr_name in ('calendar_day', 'calendar_date'):
                        raw_calendar_day = attr.get('value')
                        # Normalize date format to ensure consistent matching
                        calendar_day = normalize_date(raw_calendar_day)
                        if calendar_day:
                            break
                
                # Fallback: derive from parsed time
                if not calendar_day:
                    calendar_day = parsed_time.date().isoformat()
            
            indexed_event = IndexedEvent(
                event=event,
                event_type=event_type_name,
                parsed_time=parsed_time,
                start_time=None,
                end_time=None,
                glucose_value=glucose_value,
                start_time_s=None,
                duration_s=None,
                participant_id=participant_id,
                calendar_day=calendar_day
            )
            
            if use_calendar_day:
                key = (participant_id, calendar_day)
                if key not in index_dict:
                    index_dict[key] = []
                index_dict[key].append(indexed_event)
            else:
                if participant_id not in index_dict:
                    index_dict[participant_id] = []
                index_dict[participant_id].append(indexed_event)
        
        for key in index_dict:
            index_dict[key].sort(key=lambda e: e.parsed_time if e.parsed_time else datetime.min)
    
    def _get_event_copy(self, event: Dict[str, Any]) -> Dict[str, Any]:
        """Get a copy of an event for modification, caching the copy to avoid duplicates."""
        event_id = event.get('id')
        if not event_id:
            # If no ID, create a deep copy on the fly
            return copy.deepcopy(event)
        
        if event_id not in self._event_copies:
            self._event_copies[event_id] = copy.deepcopy(event)
        
        return self._event_copies[event_id]
    
    def _get_object_copy(self, obj: Dict[str, Any]) -> Dict[str, Any]:
        """Get a copy of an object for modification, caching the copy to avoid duplicates."""
        obj_id = obj.get('id')
        if not obj_id:
            # If no ID, create a deep copy on the fly
            return copy.deepcopy(obj)
        
        if obj_id not in self._object_copies:
            self._object_copies[obj_id] = copy.deepcopy(obj)
        
        return self._object_copies[obj_id]
    
    def _merge_inferred_data(self, enhanced_ocel: Dict[str, Any]) -> Dict[str, Any]:
        """
        Merge inferred data into enhanced OCEL, replacing original events/objects with modified copies.
        
        Args:
            enhanced_ocel: Enhanced OCEL dictionary to merge into
            
        Returns:
            Enhanced OCEL with merged data
        """
        # Replace original events with modified copies
        if self._event_copies:
            event_id_to_index = {
                event.get('id'): i
                for i, event in enumerate(enhanced_ocel.get('behaviorEvents', []))
                if event.get('id')
            }
            
            for event_id, event_copy in self._event_copies.items():
                if event_id in event_id_to_index:
                    enhanced_ocel['behaviorEvents'][event_id_to_index[event_id]] = event_copy
        
        # Replace original objects with modified copies
        if self._object_copies:
            object_id_to_index = {
                obj.get('id'): i
                for i, obj in enumerate(enhanced_ocel.get('objects', []))
                if obj.get('id')
            }
            
            for obj_id, obj_copy in self._object_copies.items():
                if obj_id in object_id_to_index:
                    enhanced_ocel['objects'][object_id_to_index[obj_id]] = obj_copy
                else:
                    # Object is not in original data, check if it's in inferred_objects
                    # and update it with the modified copy
                    for i, inferred_obj in enumerate(self.inferred_objects):
                        if inferred_obj.get('id') == obj_id:
                            self.inferred_objects[i] = obj_copy
                            break
        
        # Add inferred data
        enhanced_ocel['behaviorEvents'].extend(self.inferred_behavior_events)
        enhanced_ocel['objects'].extend(self.inferred_objects)
        
        return enhanced_ocel
    
    def infer(self) -> Dict[str, Any]:
        """
        Run inference and return enhanced OCEL data.
        
        This method should be overridden by subclasses.
        """
        raise NotImplementedError("Subclasses must implement infer() method")


class PPGRInference(BaseInference):
    """Infers PPBG objects and behavior events from blood glucose sensor events in Sensor-Augmented OCEL data."""
    
    def __init__(self, ocel_data: Dict[str, Any]):
        """
        Initialize with OCEL data.
        
        Args:
            ocel_data: Dictionary containing Sensor-Augmented OCEL data
        """
        super().__init__(ocel_data)
        
        # Counters for generating IDs
        self.ppbg_counter = 0
        self.ppic_counter = 0
        
        # Build optimized index structures
        self._build_object_index()
        
        # Build event indexes using shared helper
        self._glucose_by_participant = {}
        self._build_simple_event_index(
            events=self.sensor_events,
            event_type_name='BloodGlucose',
            event_type_field='sensorEventType',
            event_type_value='BloodGlucose',
            index_dict=self._glucose_by_participant,
            use_calendar_day=False,
            extract_glucose_value=True
        )
        
        self._meals_by_participant = {}
        self._build_simple_event_index(
            events=self.behavior_events,
            event_type_name='Meal',
            event_type_field='behaviorEventType',
            event_type_value='Meal',
            index_dict=self._meals_by_participant,
            use_calendar_day=False,
            extract_glucose_value=False
        )
        
        self._bolus_by_participant = {}
        self._build_simple_event_index(
            events=self.behavior_events,
            event_type_name='BolusInsulin',
            event_type_field='behaviorEventType',
            event_type_value='BolusInsulin',
            index_dict=self._bolus_by_participant,
            use_calendar_day=False,
            extract_glucose_value=False
        )
        
    
    def _get_event_id(self, event_type: str) -> str:
        """Generate event ID."""
        if event_type not in self.behavior_event_counter:
            self.behavior_event_counter[event_type] = 0
        self.behavior_event_counter[event_type] += 1
        abbrev = {
            'PPBGPeak': 'PPK',
            'PPBGRecovery': 'PPR'
        }.get(event_type, 'BE')
        return f"{abbrev}_{self.behavior_event_counter[event_type]}"
    
    def _event_involves_object(
        self,
        event: Dict[str, Any],
        object_id: str,
        qualifier: Optional[str] = None
    ) -> bool:
        """Check if event has a relationship to the specified object."""
        for rel in event.get('relationships', []):
            rel_obj_id = rel.get('objectId') or rel.get('id')
            if rel_obj_id != object_id:
                continue
            if qualifier and rel.get('qualifier') != qualifier:
                continue
            return True
        return False

    def _get_baseline_glucose(
        self,
        meal_time: datetime,
        participant_id: str
    ) -> Optional[float]:
        """Get baseline glucose value as average of readings in 1 hour window before meal time."""
        glucose_events = self._glucose_by_participant.get(participant_id, [])
        if not glucose_events:
            return None
        
        # Get glucose readings up to 1 hour before meal
        start_time = meal_time - timedelta(hours=1)
        
        # Collect all valid glucose values in the 1-hour window before meal
        valid_values = []
        for indexed_event in glucose_events:
            if indexed_event.parsed_time and start_time <= indexed_event.parsed_time <= meal_time:
                if indexed_event.glucose_value is not None:
                    valid_values.append(indexed_event.glucose_value)
        
        if not valid_values:
            return None
        
        # Return average of all glucose values in the 1-hour window before meal
        # This provides a more stable baseline than a single reading
        return sum(valid_values) / len(valid_values)
    
    def _get_glucose_window(
        self, 
        meal_time: datetime, 
        participant_id: str,
        next_meal_time: Optional[datetime] = None
    ) -> List[Dict[str, Any]]:
        """
        Get glucose events in window after meal (optimized with index).
        
        Args:
            meal_time: Meal event timestamp
            participant_id: Participant ID
            next_meal_time: Optional next meal time to limit window
            
        Returns:
            List of glucose sensor events sorted by time
        """
        end_time = meal_time + timedelta(hours=4)
        if next_meal_time and next_meal_time <= end_time:
            # End window just before next meal (1 minute before)
            end_time = next_meal_time - timedelta(minutes=1)
        
        glucose_events = self._glucose_by_participant.get(participant_id, [])
        if not glucose_events:
            return []
        
        # Fast time range filtering using pre-sorted list
        valid_events = []
        for indexed_event in glucose_events:
            if indexed_event.parsed_time and meal_time <= indexed_event.parsed_time <= end_time:
                if indexed_event.glucose_value is not None:
                    # Return original event (no modification needed)
                    valid_events.append(indexed_event.event)
        
        return valid_events
    
    def _get_meal_events(self, participant_id: str) -> List[IndexedEvent]:
        """Get all meal events for a participant, sorted by time (optimized with index)."""
        return self._meals_by_participant.get(participant_id, [])
    
    def _get_bolus_insulin_events(
        self,
        start_time: datetime,
        end_time: datetime,
        participant_id: str
    ) -> List[Dict[str, Any]]:
        """Get bolus insulin events in time window for participant (optimized with index)."""
        bolus_events = self._bolus_by_participant.get(participant_id, [])
        if not bolus_events:
            return []
        
        # Fast time range filtering using pre-sorted list
        valid_events = []
        for indexed_event in bolus_events:
            if indexed_event.parsed_time and start_time <= indexed_event.parsed_time <= end_time:
                valid_events.append(indexed_event.event)
        
        return valid_events

    def _iter_postprandial_windows(
        self,
        participant_id: str,
        meals: List[IndexedEvent]
    ):
        """Yield prepared postprandial windows shared by PPBG and PPIC inference."""
        total_meals = len(meals)

        for index, indexed_meal in enumerate(meals):
            if not indexed_meal.parsed_time:
                continue
            meal_time = indexed_meal.parsed_time
            meal_event = indexed_meal.event
            next_meal_time = meals[index + 1].parsed_time if index + 1 < total_meals else None

            glucose_events = self._get_glucose_window(meal_time, participant_id, next_meal_time)
            if len(glucose_events) < 2:
                continue

            baseline_value = self._get_baseline_glucose(meal_time, participant_id)
            if baseline_value is None and glucose_events:
                baseline_value = get_glucose_value(glucose_events[0])
            if baseline_value is None:
                continue

            start_time = meal_time
            if glucose_events:
                end_time = parse_time(glucose_events[-1]['time'])
            else:
                end_time = meal_time + timedelta(hours=4)

            bolus_events = self._get_bolus_insulin_events(start_time, end_time, participant_id)

            window_duration_hours = (end_time - start_time).total_seconds() / 3600

            yield {
                'participant_id': participant_id,
                'meal': meal_event,  # Return original event, not indexed
                'meal_index': index,
                'meal_count': total_meals,
                'meal_time': meal_time,
                'start_time': start_time,
                'end_time': end_time,
                'glucose_events': glucose_events,
                'baseline_value': baseline_value,
                'bolus_events': bolus_events,
                'window_duration_hours': window_duration_hours
            }

    def _associate_bolus_events(
        self,
        target_object: Dict[str, Any],
        bolus_events: List[Dict[str, Any]],
        forward_qualifier: str = 'infer',
        reverse_qualifier: str = 'inferredBy'
    ) -> None:
        """Create bidirectional associations between a target object and bolus events.
        
        Args:
            target_object: Object that infers bolus events (includes them in its window)
            bolus_events: List of bolus events inferred by the object
            forward_qualifier: Qualifier for object → event (default: 'infer')
            reverse_qualifier: Qualifier for event → object (default: 'inferredBy')
        """
        for bolus_event in bolus_events:
            # Get a copy of the bolus event to avoid modifying original
            bolus_event_copy = self._get_event_copy(bolus_event)
            if not bolus_event_copy.get('relationships'):
                bolus_event_copy['relationships'] = []
            # Event → Object: event is inferred by object
            add_relationship(
                bolus_event_copy['relationships'],
                target_object['id'],
                reverse_qualifier
            )
            # Object → Event: object infers event
            add_relationship(
                target_object['relationships'],
                bolus_event_copy['id'],
                forward_qualifier
            )

    def _create_ppbg_object(
        self,
        window: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Instantiate a PPBG object from a prepared postprandial window."""
        self.ppbg_counter += 1
        ppbg_id = f"PPBG_{self.ppbg_counter}"

        start_time: datetime = window['start_time']
        end_time: datetime = window['end_time']
        participant_id: str = window['participant_id']

        relationships: List[Dict[str, Any]] = []
        add_relationship(relationships, participant_id, 'corr')

        attributes = [
            {
                'name': 'start_time',
                'value': start_time.isoformat(),
                'time': start_time.isoformat()
            },
            {
                'name': 'end_time',
                'value': end_time.isoformat(),
                'time': end_time.isoformat()
            },
            {
                'name': 'calendar_day',
                'value': start_time.date().isoformat(),
                'time': start_time.isoformat()
            },
            {
                'name': '1h_premeal_baseline',
                'value': round_for_storage(window['baseline_value']),
                'time': window['meal_time'].isoformat()
            }
        ]

        ppbg_obj = {
            'id': ppbg_id,
            'type': 'PPBG',
            'attributes': attributes,
            'relationships': relationships
        }

        self.inferred_objects.append(ppbg_obj)

        meal = window['meal']
        if meal.get('id'):
            meal_id = meal['id']
            # PPBG → Meal: PPBG is inferred by Meal
            # Meal → PPBG: Meal infers PPBG (bidirectional)
            # Get a copy of the meal event to avoid modifying original
            meal_copy = self._get_event_copy(meal)
            if not meal_copy.get('relationships'):
                meal_copy['relationships'] = []
            # Use link_inference_relationship with meal as source (meal infers PPBG)
            link_inference_relationship(meal_copy, ppbg_obj)

        return ppbg_obj

    def _maybe_create_ppic_object(
        self,
        window: Dict[str, Any]
    ) -> Optional[Dict[str, Any]]:
        """Create a PPIC object when bolus insulin events are present."""
        bolus_events = window['bolus_events']
        if not bolus_events:
            return None

        self.ppic_counter += 1
        ppic_id = f"PPIC_{self.ppic_counter}"

        start_time: datetime = window['start_time']
        end_time: datetime = window['end_time']
        participant_id: str = window['participant_id']

        relationships: List[Dict[str, Any]] = []
        add_relationship(relationships, participant_id, 'corr')

        attributes = [
            {
                'name': 'start_time',
                'value': start_time.isoformat(),
                'time': start_time.isoformat()
            },
            {
                'name': 'end_time',
                'value': end_time.isoformat(),
                'time': end_time.isoformat()
            }
        ]

        ppic_obj = {
            'id': ppic_id,
            'type': 'PPIC',
            'attributes': attributes,
            'relationships': relationships
        }

        self.inferred_objects.append(ppic_obj)

        meal = window['meal']
        if meal.get('id'):
            meal_id = meal['id']
            # PPIC → Meal: PPIC is inferred by Meal
            # Meal → PPIC: Meal infers PPIC (bidirectional)
            # Get a copy of the meal event to avoid modifying original
            meal_copy = self._get_event_copy(meal)
            if not meal_copy.get('relationships'):
                meal_copy['relationships'] = []
            # Use link_inference_relationship with meal as source (meal infers PPIC)
            link_inference_relationship(meal_copy, ppic_obj)

        self._associate_bolus_events(ppic_obj, bolus_events, forward_qualifier='infer', reverse_qualifier='inferredBy')

        return ppic_obj
    
    def _find_maximum_peak(
        self, 
        glucose_events: List[Dict[str, Any]],
        baseline_value: float
    ) -> Tuple[Optional[Tuple[datetime, float]], Optional[int]]:
        """
        Find the absolute maximum glucose value after the meal (the peak).
        Ensures only one peak is identified per PPBG lifecycle.
        
        Args:
            glucose_events: List of glucose sensor events (sorted by time)
            baseline_value: Baseline glucose value at meal time
            
        Returns:
            Tuple of ((timestamp, value) for maximum peak, peak index) or (None, None) if no peak
        """
        if len(glucose_events) < 2:
            return None, None
        
        # Extract values and times, filtering out None values
        valid_data = []
        for i, event in enumerate(glucose_events):
            value = get_glucose_value(event)
            if value is not None:
                time = parse_time(event['time'])
                valid_data.append((i, time, value))
        
        if len(valid_data) < 2:
            return None, None
        
        # Find the absolute maximum value above baseline
        max_value = baseline_value
        max_idx = None
        max_time = None
        
        for orig_idx, time, value in valid_data:
            if value > max_value:
                max_value = value
                max_idx = orig_idx
                max_time = time
        
        # Only return a peak if we found a value above baseline
        if max_idx is None:
            return None, None
        
        return (max_time, max_value), max_idx
    
    def _find_return_to_normal(
        self,
        glucose_events: List[Dict[str, Any]],
        baseline_value: float,
        peak_index: int,
        peak_time: datetime,
        threshold: float = 0.5
    ) -> Optional[Tuple[datetime, float]]:
        """
        Find the first time glucose returns to normal after the peak (baseline ± threshold).
        Ensures only one recovery event is identified per PPBG lifecycle and that it occurs after the peak.
        
        Args:
            glucose_events: List of glucose sensor events (sorted by time)
            baseline_value: Baseline glucose value
            peak_index: Index where peak occurred
            peak_time: Timestamp of the peak event
            threshold: Tolerance for "normal" (default 0.5 mmol/L)
            
        Returns:
            (timestamp, value) tuple for the first return to normal, or None if not found
        """
        # Find first return to normal after the peak (only one)
        # Start from peak_index + 1 to ensure we're looking after the peak
        for i in range(peak_index + 1, len(glucose_events)):
            event = glucose_events[i]
            value = get_glucose_value(event)
            if value is not None:
                event_time = parse_time(event['time'])
                # Ensure recovery occurs after peak time (not just after peak index)
                if event_time > peak_time and abs(value - baseline_value) <= threshold:
                    return (event_time, value)
        return None
    
    def infer_postprandial_objects(self) -> None:
        """Infer PPBG and PPIC objects for all participants."""
        print("Starting postprandial object inference (PPBG + PPIC)...")

        # Use indexed participants instead of filtering
        participants = [p.object for p in self._participants]
        total_meals = 0
        meals_processed = 0
        meals_skipped = 0

        for participant_idx, participant in enumerate(participants, 1):
            participant_id = participant['id']
            meals = self._get_meal_events(participant_id)
            total_meals += len(meals)

            for window in self._iter_postprandial_windows(participant_id, meals):
                meals_processed += 1

                ppbg_obj = self._create_ppbg_object(window)
                self._associate_bolus_events(ppbg_obj, window['bolus_events'])

                peak_event, recovery_event, peak_value = self._infer_ppbg_behavior_events(
                    ppbg_obj,
                    window['glucose_events'],
                    window['meal_time'],
                    participant_id,
                    window['baseline_value']
                )

                excursion_value: Optional[float] = None

                if peak_event:
                    excursion_value = peak_value - window['baseline_value']
                    add_relationship(
                        ppbg_obj['relationships'],
                        peak_event['id'],
                        'infer'
                    )

                if recovery_event:
                    add_relationship(
                        ppbg_obj['relationships'],
                        recovery_event['id'],
                        'infer'
                    )

                if excursion_value is not None:
                    ppbg_obj['attributes'].append({
                        'name': 'excursion_mmolL-1',
                        'value': round_for_storage(excursion_value),
                        'time': window['end_time'].isoformat()
                    })

                ppic_obj = self._maybe_create_ppic_object(window)
                if ppic_obj:
                    add_relationship(
                        ppbg_obj['relationships'],
                        ppic_obj['id'],
                        'infer'
                    )
                    add_relationship(
                        ppic_obj['relationships'],
                        ppbg_obj['id'],
                        'infer'
                    )
                    
                    # Link recovery event to PPIC object if both exist (they are concurrent in the same window)
                    if recovery_event:
                        # PPIC → Recovery: PPIC infers Recovery (bidirectional inference)
                        # Recovery → PPIC: Recovery is inferred by PPIC
                        link_inference_relationship(ppic_obj, recovery_event)

        meals_skipped = total_meals - meals_processed
        total_ppbg = sum(1 for obj in self.inferred_objects if obj.get('type') == 'PPBG')
        total_ppic = sum(1 for obj in self.inferred_objects if obj.get('type') == 'PPIC')
        print(f"  ✓ Processed {meals_processed} meals ({meals_skipped} skipped due to insufficient data)")
        print(f"  ✓ Created {total_ppbg} PPBG object(s), {total_ppic} PPIC object(s)")

    def infer_ppbg_objects(self) -> None:
        """Backward-compatible entry point for PPBG inference."""
        self.infer_postprandial_objects()
    
    def _infer_ppbg_behavior_events(
        self,
        ppbg_obj: Dict[str, Any],
        glucose_events: List[Dict[str, Any]],
        meal_time: datetime,
        participant_id: str,
        baseline_value: float
    ) -> Tuple[Optional[Dict[str, Any]], Optional[Dict[str, Any]], Optional[float]]:
        """
        Infer PPBGPeak and PPBGRecovery behavior events for a single PPBG window.
        
        Returns:
            Tuple of (peak_event, recovery_event, peak_value)
        """
        if not glucose_events:
            return None, None, None
        
        # Find the absolute maximum peak (only one peak per PPBG)
        peak_result = self._find_maximum_peak(glucose_events, baseline_value)
        if peak_result[0] is None:
            return None, None, None

        peak_time, peak_value = peak_result[0]
        peak_idx = peak_result[1]

        peak_event = {
            'id': self._get_event_id('PPBGPeak'),
            'behaviorEventType': 'PPBGPeak',
            'time': peak_time.isoformat(),
            'behaviorEventTypeAttributes': [
                {
                    'name': 'value_mmolL-1',
                            'value': round_for_storage(peak_value)
                }
            ],
            'sensorEventTypes': [
                {
                    'sensorEventType': 'BloodGlucose',
                    'qualifier': 'inferredBy'
                }
            ],
            'relationships': []
        }
        # Link peak event to PPBG object (inference relationship)
        link_inference_relationship(ppbg_obj, peak_event)
        # Link peak event to participant (correlation)
        add_relationship(peak_event['relationships'], participant_id, 'corr')
        self.inferred_behavior_events.append(peak_event)

        recovery_event: Optional[Dict[str, Any]] = None
        # Pass peak_time to ensure recovery occurs after the peak
        back2normal = self._find_return_to_normal(glucose_events, baseline_value, peak_idx, peak_time)
        if back2normal:
            recovery_time, recovery_value = back2normal
            
            recovery_event = {
                'id': self._get_event_id('PPBGRecovery'),
                'behaviorEventType': 'PPBGRecovery',
                'time': recovery_time.isoformat(),
                'behaviorEventTypeAttributes': [
                    {
                        'name': 'value_mmolL-1',
                    'value': round_for_storage(recovery_value)
                    }
                ],
                'sensorEventTypes': [
                    {
                        'sensorEventType': 'BloodGlucose',
                        'qualifier': 'inferredBy'
                    }
                ],
                'relationships': []
            }
            # Link recovery event to PPBG object (inference relationship)
            link_inference_relationship(ppbg_obj, recovery_event)
            # Link recovery event to participant (correlation)
            add_relationship(recovery_event['relationships'], participant_id, 'corr')
            self.inferred_behavior_events.append(recovery_event)

        return peak_event, recovery_event, peak_value
    
    def add_event_types(self) -> None:
        """Add new behavior event types to OCEL data."""
        register_behavior_event_type(
            self.ocel_data,
            'PPBGPeak',
            [{'name': 'value_mmolL-1', 'type': 'number'}],
            verbose=False
        )
        register_behavior_event_type(
            self.ocel_data,
            'PPBGRecovery',
            [{'name': 'value_mmolL-1', 'type': 'number'}],
            verbose=False
        )
    
    def add_object_types(self) -> None:
        """Add PPBG and PPIC object types to OCEL data."""
        register_object_type(
            self.ocel_data,
            'PPBG',
            [
                {'name': 'start_time', 'type': 'string'},
                {'name': 'end_time', 'type': 'string'},
                {'name': 'calendar_day', 'type': 'string'},
                {'name': '1h_premeal_baseline', 'type': 'number'},
                {'name': 'excursion_mmolL-1', 'type': 'number'}
            ],
            verbose=False
        )
        register_object_type(
            self.ocel_data,
            'PPIC',
            [
                {'name': 'start_time', 'type': 'string'},
                {'name': 'end_time', 'type': 'string'}
            ],
            verbose=False
        )
    
    def infer(self) -> Dict[str, Any]:
        """
        Run inference and return enhanced OCEL data.
        
        Returns:
            Enhanced OCEL dictionary with inferred objects and events
        """
        print("\n[PPGR Inference]")
        self.add_event_types()
        self.add_object_types()
        self.infer_postprandial_objects()
        
        enhanced_ocel = self.ocel_data.copy()
        enhanced_ocel = self._merge_inferred_data(enhanced_ocel)
        
        return enhanced_ocel


class DGVDailyInference(BaseInference):
    """Infers daily glycemic variability (DGV) objects and related behavior events."""

    def __init__(self, ocel_data: Dict[str, Any]):
        super().__init__(ocel_data)
        
        self.dgv_counter = 0
        self._dgv_by_participant_day: Dict[Tuple[str, str], IndexedObject] = {}
        self._sleep_by_participant_day: Dict[Tuple[str, str], List[IndexedEvent]] = {}
        self._meal_by_participant_day: Dict[Tuple[str, str], List[IndexedEvent]] = {}

        self._build_object_index()
        
        # Build index of existing DGV objects from input
        self._build_dgv_index()
        
        # Build glucose event index using shared helper
        self._glucose_by_participant = {}
        self._build_simple_event_index(
            events=self.sensor_events,
            event_type_name='BloodGlucose',
            event_type_field='sensorEventType',
            event_type_value='BloodGlucose',
            index_dict=self._glucose_by_participant,
            use_calendar_day=False,
            extract_glucose_value=True
        )
        
        # Build sleep event index using shared helper
        self._sleep_by_participant_day = {}
        self._build_simple_event_index(
            events=self.behavior_events,
            event_type_name='SleepTime',
            event_type_field='behaviorEventType',
            event_type_value='SleepTime',
            index_dict=self._sleep_by_participant_day,
            use_calendar_day=True,
            extract_glucose_value=False
        )
        
        # Build meal event index by participant and calendar_day
        self._meal_by_participant_day = {}
        self._build_simple_event_index(
            events=self.behavior_events,
            event_type_name='Meal',
            event_type_field='behaviorEventType',
            event_type_value='Meal',
            index_dict=self._meal_by_participant_day,
            use_calendar_day=True,
            extract_glucose_value=False
        )

    def _build_dgv_index(self) -> None:
        """Map existing DGV objects by participant and day for later association (using object index)."""
        # Use indexed DGV objects from input data
        dgv_objects = self._objects_by_type.get('DGV', [])
        
        for indexed_obj in dgv_objects:
            participant_id = indexed_obj.participant_id
            calendar_day = indexed_obj.calendar_day
            
            if not participant_id or not calendar_day:
                continue
            
            # Normalize calendar_day to ensure consistent matching (should already be normalized from _build_object_index, but double-check)
            normalized_calendar_day = normalize_date(calendar_day)
            if not normalized_calendar_day:
                continue
            
            # Use normalized date for index key
            self._dgv_by_participant_day[(participant_id, normalized_calendar_day)] = indexed_obj


    @staticmethod
    def _empty_day_record() -> Dict[str, Any]:
        return {
            'total_minutes': 0.0,
            'weighted_sum': 0.0,
            'weighted_square_sum': 0.0,
            'time_in_range': 0.0,
            'time_above_range': 0.0,
            'time_below_range': 0.0,
            'raw_intervals': []
        }

    def _aggregate_daily_data(
        self,
        participant_id: str
    ) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, List[IndexedEvent]]]:
        """Aggregate glucose data per day for the participant."""
        events = self._glucose_by_participant.get(participant_id, [])
        daily_data: Dict[str, Dict[str, Any]] = {}
        day_events: Dict[str, List[IndexedEvent]] = defaultdict(list)

        if not events:
            return daily_data, day_events

        for indexed_event in events:
            if not indexed_event.parsed_time:
                continue
            day_key = indexed_event.parsed_time.date().isoformat()
            daily_data.setdefault(day_key, self._empty_day_record())
            day_events[day_key].append(indexed_event)

        for i in range(len(events) - 1):
            current_event = events[i]
            next_event = events[i + 1]

            if not current_event.parsed_time or not next_event.parsed_time:
                continue
            start_time = current_event.parsed_time
            end_time = next_event.parsed_time
            if end_time <= start_time:
                continue

            value = current_event.glucose_value
            interval_minutes_total = (end_time - start_time).total_seconds() / 60
            if interval_minutes_total <= 0:
                continue

            interval_day_key = start_time.date().isoformat()
            daily_data.setdefault(interval_day_key, self._empty_day_record())
            daily_data[interval_day_key]['raw_intervals'].append(interval_minutes_total)

            segment_start = start_time
            while segment_start < end_time:
                day_start = segment_start.replace(hour=0, minute=0, second=0, microsecond=0)
                next_day_start = day_start + timedelta(days=1)
                segment_end = min(end_time, next_day_start)

                duration_minutes = (segment_end - segment_start).total_seconds() / 60
                if duration_minutes <= 0:
                    segment_start = segment_end
                    continue

                day_key = day_start.date().isoformat()
                daily_record = daily_data.setdefault(day_key, self._empty_day_record())
                daily_record['total_minutes'] += duration_minutes
                daily_record['weighted_sum'] += value * duration_minutes
                daily_record['weighted_square_sum'] += (value ** 2) * duration_minutes

                if InferenceConfig.LOWER_THRESHOLD <= value <= InferenceConfig.UPPER_THRESHOLD:
                    daily_record['time_in_range'] += duration_minutes
                elif value > InferenceConfig.UPPER_THRESHOLD:
                    daily_record['time_above_range'] += duration_minutes
                else:
                    daily_record['time_below_range'] += duration_minutes

                segment_start = segment_end

        return daily_data, day_events

    def _compute_missing_data_percent(
        self,
        event_count: int,
        raw_intervals: List[float]
    ) -> Optional[float]:
        """Estimate missing data percentage from event counts and median interval."""
        if event_count <= 0 or not raw_intervals:
            return None

        median_interval = median(raw_intervals)
        if median_interval <= 0:
            return None

        expected_events = 1440 / median_interval
        if expected_events <= 0:
            return None

        coverage_ratio = min(event_count / expected_events, 1.0)
        missing_percent = max(0.0, 1.0 - coverage_ratio) * 100
        return round_for_storage(missing_percent, ndigits=2)

    def _compute_metrics(
        self,
        day_record: Dict[str, Any],
        day_events: List[IndexedEvent]
    ) -> Dict[str, Optional[float]]:
        """Compute daily metrics for a DGV object."""
        total_minutes = day_record['total_minutes']

        if total_minutes > 0:
            mean_value = day_record['weighted_sum'] / total_minutes
            variance = (day_record['weighted_square_sum'] / total_minutes) - (mean_value ** 2)
            variance = max(variance, 0.0)
            sd_value = math.sqrt(variance)
            cv_percent = (sd_value / mean_value * 100) if mean_value else None

            time_in_range_minutes = day_record['time_in_range']
            time_above_minutes = day_record['time_above_range']
            time_below_minutes = day_record['time_below_range']

            time_in_range_percent = (time_in_range_minutes / total_minutes * 100) if total_minutes else None
            time_above_percent = (time_above_minutes / total_minutes * 100) if total_minutes else None
            time_below_percent = (time_below_minutes / total_minutes * 100) if total_minutes else None
        else:
            values = [e.glucose_value for e in day_events if e.glucose_value is not None]
            if values:
                mean_value = sum(values) / len(values)
                variance = sum((v - mean_value) ** 2 for v in values) / len(values) if len(values) > 1 else 0.0
                sd_value = math.sqrt(max(variance, 0.0))
                cv_percent = (sd_value / mean_value * 100) if mean_value else None
            else:
                mean_value = None
                sd_value = None
                cv_percent = None

            time_in_range_minutes = 0.0
            time_above_minutes = 0.0
            time_below_minutes = 0.0
            time_in_range_percent = None
            time_above_percent = None
            time_below_percent = None

        return {
            'mean': round_for_storage(mean_value),
            'sd': round_for_storage(sd_value),
            'cv': round_for_storage(cv_percent),
            'time_in_range_minutes': round_for_storage(time_in_range_minutes),
            'time_in_range_percent': round_for_storage(time_in_range_percent),
            'time_above_minutes': round_for_storage(time_above_minutes),
            'time_above_percent': round_for_storage(time_above_percent),
            'time_below_minutes': round_for_storage(time_below_minutes),
            'time_below_percent': round_for_storage(time_below_percent)
        }

    def infer_dgv_objects(self) -> None:
        """Infer DGV objects per participant per day."""
        # Use indexed participants instead of filtering
        participants = [p.object for p in self._participants]
        print("Starting DGV object inference...")

        for participant_idx, participant in enumerate(participants, 1):
            participant_id = participant['id']
            daily_data, day_events = self._aggregate_daily_data(participant_id)
            if not daily_data:
                continue

            for day_key in sorted(daily_data.keys()):
                day_record = daily_data[day_key]
                events_for_day = day_events.get(day_key, [])
                metrics = self._compute_metrics(day_record, events_for_day)
                missing_data_percent = self._compute_missing_data_percent(len(events_for_day), day_record['raw_intervals'])

                self.dgv_counter += 1
                dgv_id = f"DGV_{self.dgv_counter}"

                # Normalize day_key to ensure consistent date format
                normalized_day_key = normalize_date(day_key)
                if not normalized_day_key:
                    continue

                if len(normalized_day_key) == 10:
                    attribute_time_iso = datetime.fromisoformat(normalized_day_key).isoformat()
                else:
                    attribute_time_iso = parse_time(normalized_day_key).isoformat()

                attributes = [
                    {
                        'name': 'calendar_day',
                        'value': normalized_day_key,
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'mean_glucose_mmolL-1',
                        'value': metrics['mean'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'SDMG_mmolL-1',
                        'value': metrics['sd'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'CV',
                        'value': metrics['cv'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'time_in_range_minutes',
                        'value': metrics['time_in_range_minutes'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'time_in_range_percent',
                        'value': metrics['time_in_range_percent'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'time_above_range_minutes',
                        'value': metrics['time_above_minutes'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'time_above_range_percent',
                        'value': metrics['time_above_percent'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'time_below_range_minutes',
                        'value': metrics['time_below_minutes'],
                        'time': attribute_time_iso
                    },
                    {
                        'name': 'time_below_range_percent',
                        'value': metrics['time_below_percent'],
                        'time': attribute_time_iso
                    }
                ]

                attributes.append({
                    'name': 'missing_data_percent',
                    'value': missing_data_percent,
                    'time': attribute_time_iso
                })

                # Skip creating DGV object if one already exists for this participant-day
                if (participant_id, normalized_day_key) in self._dgv_by_participant_day:
                    continue
                
                relationships: List[Dict[str, Any]] = []
                add_relationship(relationships, participant_id, 'corr')

                dgv_object = {
                    'id': dgv_id,
                    'type': 'DGV',
                    'attributes': attributes,
                    'relationships': relationships
                }

                self.inferred_objects.append(dgv_object)
                
                # Create IndexedObject for DGV object and add to index (use normalized date)
                indexed_dgv = IndexedObject(
                    object=dgv_object,
                    object_type='DGV',
                    participant_id=participant_id,
                    calendar_day=normalized_day_key
                )
                self._dgv_by_participant_day[(participant_id, normalized_day_key)] = indexed_dgv
                
                # Link Meal events to DGV object for the same day
                self._link_meal_events_to_dgv(dgv_object, participant_id, normalized_day_key)

        print(f"  ✓ Created {len(self.inferred_objects)} DGV object(s)")

    def _link_meal_events_to_dgv(self, dgv_object: Dict[str, Any], participant_id: str, calendar_day: str) -> None:
        """Create bidirectional relationships between DGV object and Meal events for the same day.
        
        DGV and Meal events are correlated (same day), so:
        - DGV → Meal: 'infer' (DGV infers Meal)
        - Meal → DGV: 'inferredBy' (Meal is inferred by DGV)
        """
        # Normalize calendar_day to ensure consistent matching
        normalized_calendar_day = normalize_date(calendar_day)
        if not normalized_calendar_day:
            return
        
        key = (participant_id, normalized_calendar_day)
        meal_events = self._meal_by_participant_day.get(key, [])
        
        for indexed_meal in meal_events:
            # Get a copy of the meal event to avoid modifying original
            meal_event = self._get_event_copy(indexed_meal.event)
            if not meal_event.get('relationships'):
                meal_event['relationships'] = []
            # Meal → DGV: event is inferred by DGV object
            # DGV → Meal: DGV object infers event
            link_inference_relationship(dgv_object, meal_event)

    def _link_sleep_events_to_dgv(self) -> None:
        """Link all existing DGV objects to SleepTime events for the same participant-day."""
        total_links = 0
        total_dgv_objects = len(self._dgv_by_participant_day)
        
        for (participant_id, calendar_day), indexed_dgv in self._dgv_by_participant_day.items():
            # Normalize calendar_day to ensure consistent matching
            normalized_calendar_day = normalize_date(calendar_day)
            if not normalized_calendar_day:
                continue
            
            # Get a copy of the DGV object to avoid modifying original
            dgv_object = self._get_object_copy(indexed_dgv.object)
            if not dgv_object.get('relationships'):
                dgv_object['relationships'] = []
            
            # Link to SleepTime events for the same day (use normalized date for lookup)
            key = (participant_id, normalized_calendar_day)
            sleep_events = self._sleep_by_participant_day.get(key, [])
            total_links += len(sleep_events)
            
            for indexed_sleep in sleep_events:
                # Get a copy of the sleep event to avoid modifying original
                sleep_event = self._get_event_copy(indexed_sleep.event)
                if not sleep_event.get('relationships'):
                    sleep_event['relationships'] = []
                # DGV → SleepTime: 'infer' (DGV infers SleepTime)
                # SleepTime → DGV: 'inferredBy' (SleepTime is inferred by DGV)
                link_inference_relationship(dgv_object, sleep_event)
            
            # Update the indexed DGV object with the modified copy
            updated_indexed_dgv = IndexedObject(
                object=dgv_object,
                object_type=indexed_dgv.object_type,
                participant_id=indexed_dgv.participant_id,
                calendar_day=indexed_dgv.calendar_day
            )
            self._dgv_by_participant_day[(participant_id, calendar_day)] = updated_indexed_dgv
            
            # Also update the object in inferred_objects if it's a newly created object
            # This ensures the modified copy with relationships is directly in inferred_objects
            dgv_id = dgv_object.get('id')
            if dgv_id:
                for i, inferred_obj in enumerate(self.inferred_objects):
                    if inferred_obj.get('id') == dgv_id:
                        # Update with the modified copy that has relationships
                        self.inferred_objects[i] = dgv_object
                        break
        
        if total_links > 0:
            print(f"  ✓ Linked {total_links} SleepTime event(s) to {total_dgv_objects} DGV object(s)")

    def _next_behavior_event_id(self, event_type: str) -> str:
        self.behavior_event_counter[event_type] += 1
        abbrev_map = {
            'HypoglycemiaStart': 'HypoGS',
            'HypoglycemiaEnd': 'HypoGE',
            'HyperglycemiaStart': 'HyperGS',
            'HyperglycemiaEnd': 'HyperGE'
        }
        prefix = abbrev_map.get(event_type, 'DGV')
        return f"{prefix}_{self.behavior_event_counter[event_type]}"

    def _find_concurrent_ppbg_ppic(
        self,
        timestamp: datetime,
        participant_id: str
    ) -> List[Dict[str, Any]]:
        """Find PPBG/PPIC objects that are concurrent with the given timestamp."""
        concurrent_objects = []
        
        # Use existing object index by type
        ppbg_objects = self._objects_by_type.get('PPBG', [])
        ppic_objects = self._objects_by_type.get('PPIC', [])
        
        for indexed_obj in ppbg_objects + ppic_objects:
            # Check if object belongs to the same participant
            if indexed_obj.participant_id != participant_id:
                continue
            
            obj = indexed_obj.object
            # Extract start_time and end_time from attributes
            start_time = None
            end_time = None
            for attr in obj.get('attributes', []):
                if attr.get('name') == 'start_time':
                    try:
                        start_time = parse_time(attr.get('value'))
                    except (ValueError, TypeError):
                        pass
                elif attr.get('name') == 'end_time':
                    try:
                        end_time = parse_time(attr.get('value'))
                    except (ValueError, TypeError):
                        pass
            
            # Check if event timestamp falls within the PPBG/PPIC object's time window
            # (event happens between start_time and end_time, inclusive)
            if start_time and end_time and start_time <= timestamp <= end_time:
                concurrent_objects.append(obj)
        
        return concurrent_objects

    def _create_behavior_event(
        self,
        event_type: str,
        timestamp: datetime,
        participant_id: str,
        dgv_object: Dict[str, Any]
    ) -> None:
        event_id = self._next_behavior_event_id(event_type)
        event = {
            'id': event_id,
            'behaviorEventType': event_type,
            'time': timestamp.isoformat(),
            'behaviorEventTypeAttributes': [],
            'sensorEventTypes': [
                {
                    'sensorEventType': 'BloodGlucose',
                    'qualifier': 'inferredBy'
                }
            ],
            'relationships': []
        }

        # Link event to DGV object (inference relationship)
        link_inference_relationship(dgv_object, event)
        # Link event to participant (correlation)
        add_relationship(event['relationships'], participant_id, 'corr')

        # Link Hypoglycemia and Hyperglycemia events to concurrent PPBG/PPIC objects
        # Check if event timestamp falls within the time window of any PPBG/PPIC object
        if event_type in ('HypoglycemiaStart', 'HypoglycemiaEnd', 'HyperglycemiaStart', 'HyperglycemiaEnd'):
            concurrent_objects = self._find_concurrent_ppbg_ppic(timestamp, participant_id)
            for obj in concurrent_objects:
                # Get a copy of the object to avoid modifying original
                obj_copy = self._get_object_copy(obj)
                if not obj_copy.get('relationships'):
                    obj_copy['relationships'] = []
                # Bidirectional inference: Hypo/Hyperglycemia event is inferred by PPBG/PPIC
                link_inference_relationship(obj_copy, event)

        self.inferred_behavior_events.append(event)

    def _classify_value(self, value: float) -> Optional[str]:
        if value < InferenceConfig.LOWER_THRESHOLD:
            return 'hypo'
        if value > InferenceConfig.UPPER_THRESHOLD:
            return 'hyper'
        return None

    def infer_behavior_events(self) -> None:
        """Infer behavior events for hypo/hyperglycemia episodes."""
        print("Starting DGV behavior event inference...")

        for participant_id, events in self._glucose_by_participant.items():
            if not events:
                continue

            state: Optional[str] = None
            last_event: Optional[IndexedEvent] = None

            for indexed_event in events:
                value = indexed_event.glucose_value
                if value is None or not indexed_event.parsed_time:
                    continue

                classification = self._classify_value(value)
                day_key = indexed_event.parsed_time.date().isoformat()
                indexed_dgv = self._dgv_by_participant_day.get((participant_id, day_key))

                if not indexed_dgv:
                    continue

                dgv_object = indexed_dgv.object
                parsed_time = indexed_event.parsed_time

                if classification == 'hypo':
                    if state != 'hypo':
                        self._create_behavior_event(
                            'HypoglycemiaStart',
                            parsed_time,
                            participant_id,
                            dgv_object
                        )
                        state = 'hypo'
                elif classification == 'hyper':
                    if state != 'hyper':
                        self._create_behavior_event(
                            'HyperglycemiaStart',
                            parsed_time,
                            participant_id,
                            dgv_object
                        )
                        state = 'hyper'
                else:
                    if state == 'hypo':
                        self._create_behavior_event(
                            'HypoglycemiaEnd',
                            parsed_time,
                            participant_id,
                            dgv_object
                        )
                        state = None
                    elif state == 'hyper':
                        self._create_behavior_event(
                            'HyperglycemiaEnd',
                            parsed_time,
                            participant_id,
                            dgv_object
                        )
                        state = None

                last_event = indexed_event

            if state and last_event and last_event.parsed_time:
                day_key = last_event.parsed_time.date().isoformat()
                indexed_dgv = self._dgv_by_participant_day.get((participant_id, day_key))
                if indexed_dgv:
                    dgv_object = indexed_dgv.object
                    closing_event_type = 'HypoglycemiaEnd' if state == 'hypo' else 'HyperglycemiaEnd'
                    self._create_behavior_event(
                        closing_event_type,
                        last_event.parsed_time,
                        participant_id,
                        dgv_object
                    )

        print(f"  ✓ Created {len(self.inferred_behavior_events)} DGV behavior event(s)")

    def add_event_types(self) -> None:
        """Add behavior event types for DGV inference."""
        required_types = [
            'HypoglycemiaStart',
            'HypoglycemiaEnd',
            'HyperglycemiaStart',
            'HyperglycemiaEnd'
        ]

        for event_type in required_types:
            register_behavior_event_type(
                self.ocel_data,
                event_type,
                [],
                verbose=False
            )

    def add_object_type(self) -> None:
        """Add DGV object type to OCEL data."""
        register_object_type(
            self.ocel_data,
            'DGV',
            [
                {'name': 'calendar_day', 'type': 'string'},
                {'name': 'mean_glucose_mmolL-1', 'type': 'number'},
                {'name': 'SDMG_mmolL-1', 'type': 'number'},
                {'name': 'CV', 'type': 'number'},
                {'name': 'time_in_range_minutes', 'type': 'number'},
                {'name': 'time_in_range_percent', 'type': 'number'},
                {'name': 'time_above_range_minutes', 'type': 'number'},
                {'name': 'time_above_range_percent', 'type': 'number'},
                {'name': 'time_below_range_minutes', 'type': 'number'},
                {'name': 'time_below_range_percent', 'type': 'number'},
                {'name': 'missing_data_percent', 'type': 'number'}
            ],
            verbose=False
        )

    def infer(self) -> Dict[str, Any]:
        """Run DGV inference and return enhanced data."""
        print("\n[DGV Inference]")
        self.add_event_types()
        self.add_object_type()
        self.infer_dgv_objects()
        self._link_sleep_events_to_dgv()
        self.infer_behavior_events()

        enhanced_ocel = self.ocel_data.copy()
        enhanced_ocel = self._merge_inferred_data(enhanced_ocel)

        return enhanced_ocel


class DADailyInference(BaseInference):
    """Infers Daily Activity (DA) objects and supporting behavior events."""

    def __init__(self, ocel_data: Dict[str, Any]):
        super().__init__(ocel_data)
        
        self.da_counter = 0
        self._dgv_by_participant_day: Dict[Tuple[str, str], IndexedObject] = {}
        self._sleep_by_participant_day: Dict[Tuple[str, str], List[IndexedEvent]] = {}
        self._meal_by_participant_day: Dict[Tuple[str, str], List[IndexedEvent]] = {}
        self._activity_by_participant_day: Dict[str, Dict[str, List[IndexedEvent]]] = {}
        self._da_by_participant_day: Dict[Tuple[str, str], Dict[str, Any]] = {}

        self._build_object_index()
        self._build_dgv_index()
        
        # Build sleep event index using shared helper
        self._sleep_by_participant_day = {}
        self._build_simple_event_index(
            events=self.behavior_events,
            event_type_name='SleepTime',
            event_type_field='behaviorEventType',
            event_type_value='SleepTime',
            index_dict=self._sleep_by_participant_day,
            use_calendar_day=True,
            extract_glucose_value=False
        )
        
        # Build meal event index by participant and calendar_day
        self._meal_by_participant_day = {}
        self._build_simple_event_index(
            events=self.behavior_events,
            event_type_name='Meal',
            event_type_field='behaviorEventType',
            event_type_value='Meal',
            index_dict=self._meal_by_participant_day,
            use_calendar_day=True,
            extract_glucose_value=False
        )
        
        self._build_activity_index()
    
    def _build_dgv_index(self) -> None:
        """Map DGV objects by participant and day for later association (using object index)."""
        # Use indexed DGV objects instead of filtering all objects
        dgv_objects = self._objects_by_type.get('DGV', [])
        
        for indexed_obj in dgv_objects:
            participant_id = indexed_obj.participant_id
            calendar_day = indexed_obj.calendar_day
            
            if not participant_id or not calendar_day:
                continue
            
            # Use the IndexedObject directly (already has all needed fields)
            self._dgv_by_participant_day[(participant_id, calendar_day)] = indexed_obj

    @staticmethod
    def _get_behavior_attribute(event: Dict[str, Any], attribute_name: str) -> Optional[Any]:
        for attr in event.get('behaviorEventTypeAttributes', []):
            if attr.get('name') == attribute_name:
                return attr.get('value')
        return None

    @staticmethod
    def _to_float(value: Any) -> float:
        try:
            return float(value)
        except (TypeError, ValueError):
            return 0.0

    def _build_activity_index(self) -> None:
        """Index Activity events by participant and calendar day, sorted by start_time."""
        self._activity_by_participant_day = {}

        for event in self.behavior_events:
            if event.get('behaviorEventType') != 'Activity':
                continue

            participant_id = get_participant_id(event, self.object_map)
            if not participant_id:
                continue

            # Get start time - prefer event's 'time' attribute (ISO format), fallback to start_time_s (epoch)
            time_str = event.get('time')
            start_time_seconds = self._to_float(self._get_behavior_attribute(event, 'start_time_s'))
            
            if time_str:
                # Use the event's time attribute (ISO format) as primary source
                try:
                    start_time = parse_time(time_str)
                except (ValueError, AttributeError, TypeError):
                    # If parsing fails, try start_time_s as epoch
                    if start_time_seconds > 0:
                        try:
                            if start_time_seconds > 1e10:  # Likely milliseconds
                                start_time = datetime.fromtimestamp(start_time_seconds / 1000.0)
                            else:
                                start_time = datetime.fromtimestamp(start_time_seconds)
                        except (ValueError, OSError):
                            continue
                    else:
                        continue
            elif start_time_seconds > 0:
                # Fallback: use start_time_s as epoch timestamp
                # Check if it's in milliseconds (epoch > year 2286) or seconds
                try:
                    if start_time_seconds > 1e10:  # Likely milliseconds (epoch > year 2286)
                        start_time = datetime.fromtimestamp(start_time_seconds / 1000.0)
                    else:
                        start_time = datetime.fromtimestamp(start_time_seconds)
                except (ValueError, OSError):
                    continue
            else:
                # No valid timestamp available
                continue

            calendar_day = start_time.date().isoformat()

            # Get duration and calculate end time
            duration_seconds = self._to_float(self._get_behavior_attribute(event, 'duration_s'))
            end_time = start_time + timedelta(seconds=duration_seconds)

            # Store indexed event data separately (no event modification)
            indexed_event = IndexedEvent(
                event=event,
                event_type='Activity',
                parsed_time=None,
                start_time=start_time,
                end_time=end_time,
                glucose_value=None,
                start_time_s=start_time_seconds,
                duration_s=duration_seconds,
                participant_id=participant_id,
                calendar_day=calendar_day
            )

            self._activity_by_participant_day.setdefault(participant_id, {}).setdefault(calendar_day, []).append(indexed_event)

        # Sort entries by start_time for each participant-day
        for participant_data in self._activity_by_participant_day.values():
            for day_entries in participant_data.values():
                day_entries.sort(key=lambda entry: entry.start_time if entry.start_time else datetime.min)

    def infer_da_objects(self) -> None:
        """Create DA objects per participant per day based on activity, sleep events, or DGV objects."""
        # Build index of existing DA objects to avoid duplicates and for later lookups
        existing_da_by_participant_day: Dict[Tuple[str, str], Dict[str, Any]] = {}
        for indexed_obj in self._objects_by_type.get('DA', []):
            participant_id = indexed_obj.participant_id
            calendar_day = indexed_obj.calendar_day
            if participant_id and calendar_day:
                existing_da_by_participant_day[(participant_id, calendar_day)] = indexed_obj.object
                # Also index in _da_by_participant_day for later lookups (e.g., in infer_pa_behavior_events)
                self._da_by_participant_day[(participant_id, calendar_day)] = indexed_obj.object
        
        # Collect all unique participant-day combinations from activity, sleep events, and DGV objects
        participant_days = set()
        
        # Add days from activity events
        for participant_id, day_map in self._activity_by_participant_day.items():
            for calendar_day in day_map.keys():
                participant_days.add((participant_id, calendar_day))
        
        # Add days from sleep events
        for (participant_id, calendar_day) in self._sleep_by_participant_day.keys():
            participant_days.add((participant_id, calendar_day))
        
        # Add days from DGV objects
        for (participant_id, calendar_day) in self._dgv_by_participant_day.keys():
            participant_days.add((participant_id, calendar_day))
        
        # Create DA objects for each participant-day combination (skip if already exists)
        for participant_id, calendar_day in sorted(participant_days):
            # Skip if DA object already exists for this participant-day
            if (participant_id, calendar_day) in existing_da_by_participant_day:
                continue
            
            try:
                day_date = datetime.fromisoformat(calendar_day).date()
                start_time = datetime.combine(day_date, datetime.min.time())
            except:
                start_time = datetime.now()
            
            # Calculate daily totals from all Activity events
            daily_totals = self._calculate_daily_activity_totals(participant_id, calendar_day)
            
            da_object = self._create_da_object(participant_id, calendar_day, start_time, daily_totals)
            # Index DA object for O(1) lookup
            self._da_by_participant_day[(participant_id, calendar_day)] = da_object
            self._link_sleep_events(da_object, participant_id, calendar_day)
            self._link_activity_events(da_object, participant_id, calendar_day)
            self._link_meal_events(da_object, participant_id, calendar_day)
            self._link_dgv_object(da_object, participant_id, calendar_day)

    def _calculate_daily_activity_totals(self, participant_id: str, calendar_day: str) -> Dict[str, float]:
        """Calculate daily totals from all Activity events for a participant-day."""
        totals = {
            'total_active_kcal': 0.0,
            'total_step_count': 0.0,
            'total_distance_m': 0.0,
            'total_duration_s': 0.0
        }
        
        day_entries = self._activity_by_participant_day.get(participant_id, {}).get(calendar_day, [])
        
        for indexed_event in day_entries:
            event = indexed_event.event
            
            # Sum all activity metrics
            totals['total_active_kcal'] += self._to_float(self._get_behavior_attribute(event, 'activity_Kcal'))
            totals['total_step_count'] += self._to_float(self._get_behavior_attribute(event, 'step_count'))
            totals['total_distance_m'] += self._to_float(self._get_behavior_attribute(event, 'distance_m'))
            totals['total_duration_s'] += indexed_event.duration_s if indexed_event.duration_s else 0.0
        
        return totals

    def _create_da_object(
        self, 
        participant_id: str, 
        calendar_day: str, 
        start_time: datetime,
        daily_totals: Dict[str, float]
    ) -> Dict[str, Any]:
        self.da_counter += 1
        da_id = f"DA_{self.da_counter}"

        relationships: List[Dict[str, Any]] = []
        add_relationship(relationships, participant_id, 'corr')

        # Calculate end time for duration attribute
        end_time = start_time + timedelta(seconds=daily_totals['total_duration_s'])

        attributes = [
            {
                'name': 'calendar_day',
                'value': calendar_day,
                'time': start_time.isoformat()
            },
            {
                'name': 'total_active_kcal',
                'value': round_for_storage(daily_totals['total_active_kcal']),
                'time': start_time.isoformat()
            },
            {
                'name': 'total_step_count',
                'value': round_for_storage(daily_totals['total_step_count']),
                'time': start_time.isoformat()
            },
            {
                'name': 'total_distance_m',
                'value': round_for_storage(daily_totals['total_distance_m']),
                'time': start_time.isoformat()
            },
            {
                'name': 'total_duration_s',
                'value': round_for_storage(daily_totals['total_duration_s']),
                'time': end_time.isoformat()
            }
        ]

        da_object = {
            'id': da_id,
            'type': 'DA',
            'attributes': attributes,
            'relationships': relationships
        }

        self.inferred_objects.append(da_object)
        return da_object

    def _link_sleep_events(self, da_object: Dict[str, Any], participant_id: str, calendar_day: str) -> None:
        """Create bidirectional relationships between DA object and SleepTime events for the same day.
        
        DA infers SleepTime events (daily aggregation), so:
        - DA → SleepTime: 'infer' (DA infers SleepTime)
        - SleepTime → DA: 'inferredBy' (SleepTime is inferred by DA)
        """
        key = (participant_id, calendar_day)
        sleep_events = self._sleep_by_participant_day.get(key, [])
        
        for indexed_sleep in sleep_events:
            # Get a copy of the sleep event to avoid modifying original
            sleep_event = self._get_event_copy(indexed_sleep.event)
            if not sleep_event.get('relationships'):
                sleep_event['relationships'] = []
            # SleepTime → DA: event is inferred by DA object
            # DA → SleepTime: DA object infers event
            link_inference_relationship(da_object, sleep_event)

    def _link_meal_events(self, da_object: Dict[str, Any], participant_id: str, calendar_day: str) -> None:
        """Create bidirectional relationships between DA object and Meal events for the same day.
        
        DA and Meal events are correlated (same day), so:
        - DA → Meal: 'infer' (DA infers Meal)
        - Meal → DA: 'inferredBy' (Meal is inferred by DA)
        """
        key = (participant_id, calendar_day)
        meal_events = self._meal_by_participant_day.get(key, [])
        
        for indexed_meal in meal_events:
            # Get a copy of the meal event to avoid modifying original
            meal_event = self._get_event_copy(indexed_meal.event)
            if not meal_event.get('relationships'):
                meal_event['relationships'] = []
            # Meal → DA: event is inferred by DA object
            # DA → Meal: DA object infers event
            link_inference_relationship(da_object, meal_event)

    def _link_activity_events(self, da_object: Dict[str, Any], participant_id: str, calendar_day: str) -> None:
        """Create bidirectional relationships between DA object and all Activity events for the same day.
        
        DA infers Activity events (daily aggregation), so:
        - DA → Activity: 'infer' (DA infers Activity)
        - Activity → DA: 'inferredBy' (Activity is inferred by DA)
        """
        day_entries = self._activity_by_participant_day.get(participant_id, {}).get(calendar_day, [])
        
        for indexed_event in day_entries:
            # Get a copy of the activity event to avoid modifying original
            activity_event = self._get_event_copy(indexed_event.event)
            if not activity_event.get('relationships'):
                activity_event['relationships'] = []
            # Activity → DA: event is inferred by DA object
            # DA → Activity: DA object infers event
            link_inference_relationship(da_object, activity_event)

    def _link_dgv_object(self, da_object: Dict[str, Any], participant_id: str, calendar_day: str) -> None:
        """Create bidirectional 'infer' relationship between DA object and DGV object for the same day.
        
        DA and DGV infer each other (same temporal window), so:
        - DA → DGV: 'infer' (DA infers DGV)
        - DGV → DA: 'infer' (DGV infers DA)
        """
        indexed_dgv = self._dgv_by_participant_day.get((participant_id, calendar_day))
        if indexed_dgv:
            # Get a copy of the DGV object to avoid modifying original
            dgv_object = self._get_object_copy(indexed_dgv.object)
            if not dgv_object.get('relationships'):
                dgv_object['relationships'] = []
            # DA ↔ DGV: bidirectional inference relationship
            create_bidirectional_relationship(da_object, dgv_object, 'infer', 'infer')
            
            # Update the indexed DGV object with the modified copy
            updated_indexed_dgv = IndexedObject(
                object=dgv_object,
                object_type=indexed_dgv.object_type,
                participant_id=indexed_dgv.participant_id,
                calendar_day=indexed_dgv.calendar_day
            )
            self._dgv_by_participant_day[(participant_id, calendar_day)] = updated_indexed_dgv

    def infer_pa_behavior_events(self) -> None:
        """Infer PAStart and PAEnd behavior events from Activity events."""
        print("Inferring PA behavior events from Activity events...")
        
        for participant_id, day_map in self._activity_by_participant_day.items():
            for calendar_day, entries in day_map.items():
                # Filter qualifying activities: active_time_s > 120 and intensity != "SEDENTARY"
                qualifying_entries = []
                for indexed_event in entries:
                    event = indexed_event.event
                    active_time_s = self._to_float(self._get_behavior_attribute(event, 'active_time_s'))
                    intensity = self._get_behavior_attribute(event, 'intensity')
                    
                    if active_time_s > 120:
                        intensity_str = str(intensity).strip().upper() if intensity else ""
                        if intensity_str != 'SEDENTARY':
                            qualifying_entries.append(indexed_event)
                
                if not qualifying_entries:
                    continue
                
                # Create individual PA events for qualifying activities
                individual_pa_events = []
                for indexed_event in qualifying_entries:
                    if not indexed_event.start_time or not indexed_event.end_time:
                        continue
                    start_event, end_event = self._create_pa_events_from_activity(
                        indexed_event.event, 
                        indexed_event.start_time, 
                        indexed_event.end_time
                    )
                    individual_pa_events.append({
                        'start_event': start_event,
                        'end_event': end_event,
                        'indexed_event': indexed_event,
                        'active_time_s': self._to_float(self._get_behavior_attribute(indexed_event.event, 'active_time_s'))
                    })
                
                # Merge consecutive timeslots where both have active_time_s > 600
                merged_pa_events = self._merge_high_activity_periods(individual_pa_events)
                
                # Add all PA events (merged and individual) to inferred events
                for pa_pair in merged_pa_events:
                    # Link to DA object if exists (O(1) lookup using index)
                    da_object = self._da_by_participant_day.get((participant_id, calendar_day))
                    if da_object:
                        # PA events are inferred by DA objects (keep only 'inferredBy', remove 'corr')
                        add_relationship(pa_pair['start_event']['relationships'], da_object['id'], 'inferredBy')
                        add_relationship(pa_pair['end_event']['relationships'], da_object['id'], 'inferredBy')
                        add_relationship(da_object['relationships'], pa_pair['start_event']['id'], 'infer')
                        add_relationship(da_object['relationships'], pa_pair['end_event']['id'], 'infer')
                    
                    # Note: PA events are NOT related to DGV objects - they come from different data sources
                    # (PA from Activity events, DGV from glucose data) and are only correlated through time
                    
                    self.inferred_behavior_events.append(pa_pair['start_event'])
                    self.inferred_behavior_events.append(pa_pair['end_event'])
        
        print(f"  ✓ Created {len(self.inferred_behavior_events)} PA behavior event(s)")

    def _create_pa_events_from_activity(
        self,
        activity_event: Dict[str, Any],
        start_time: datetime,
        end_time: datetime
    ) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """Create PAStart and PAEnd events from an Activity event with all original attributes."""
        # Copy all attributes from original activity event
        event_attributes = []
        for attr in activity_event.get('behaviorEventTypeAttributes', []):
            event_attributes.append({
                'name': attr['name'],
                'value': attr['value'],
                'time': start_time.isoformat() if attr['name'] != 'duration_s' else end_time.isoformat()
            })
        
        # Generate event IDs
        start_event_id = self._next_behavior_event_id('PAStart')
        end_event_id = self._next_behavior_event_id('PAEnd')
        
        # Create start event
        start_event = {
            'id': start_event_id,
            'behaviorEventType': 'PAStart',
            'time': start_time.isoformat(),
            'behaviorEventTypeAttributes': event_attributes,
            'sensorEventTypes': activity_event.get('sensorEventTypes', []).copy(),
            'relationships': []
        }
        
        # Create end event (same attributes, but time is end_time)
        end_event_attributes = []
        for attr in activity_event.get('behaviorEventTypeAttributes', []):
            end_event_attributes.append({
                'name': attr['name'],
                'value': attr['value'],
                'time': end_time.isoformat()
            })
        
        end_event = {
            'id': end_event_id,
            'behaviorEventType': 'PAEnd',
            'time': end_time.isoformat(),
            'behaviorEventTypeAttributes': end_event_attributes,
            'sensorEventTypes': activity_event.get('sensorEventTypes', []).copy(),
            'relationships': []
        }
        
        # Copy relationships from original activity event (except we'll add our own)
        participant_id = get_participant_id(activity_event, self.object_map)
        if participant_id:
            add_relationship(start_event['relationships'], participant_id, 'corr')
            add_relationship(end_event['relationships'], participant_id, 'corr')
        
        # Link to original activity event (bidirectional with inverse qualifiers)
        # PA events are inferred from Activity events, so:
        # - DA → PA: 'infer' (DA infers PA from Activity)
        # - PA → Activity: 'inferredBy' (PA is inferred by Activity - Activity is source)
        # - Activity → PA: 'infer' (Activity infers PA - bidirectional)
        if not activity_event.get('relationships'):
            activity_event['relationships'] = []
        # PA → Activity: PA events are inferred by Activity (Activity is the source)
        add_relationship(start_event['relationships'], activity_event['id'], 'inferredBy')
        add_relationship(end_event['relationships'], activity_event['id'], 'inferredBy')
        # Activity → PA: Activity infers PA (bidirectional)
        add_relationship(activity_event['relationships'], start_event_id, 'infer')
        add_relationship(activity_event['relationships'], end_event_id, 'infer')
        
        return start_event, end_event

    def _merge_high_activity_periods(
        self,
        individual_pa_events: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """Merge consecutive PA events where both have active_time_s > 600."""
        if not individual_pa_events:
            return []
        
        # Sort by start time
        sorted_events = sorted(individual_pa_events, key=lambda x: x['indexed_event'].start_time if x['indexed_event'].start_time else datetime.min)
        
        merged_events = []
        i = 0
        
        while i < len(sorted_events):
            current = sorted_events[i]
            current_active_time = current['active_time_s']
            
            # Check if current event qualifies for merging (active_time_s > 600)
            if current_active_time > 600:
                # Look for consecutive events to merge
                merge_group = [current]
                j = i + 1
                
                while j < len(sorted_events):
                    next_event = sorted_events[j]
                    next_active_time = next_event['active_time_s']
                    
                    # Get the last event in merge_group to check consecutiveness
                    last_in_group = merge_group[-1]
                    
                    # Check if next event is consecutive (starts at or before last event ends)
                    # and both have active_time_s > 600
                    next_start = next_event['indexed_event'].start_time
                    last_end = last_in_group['indexed_event'].end_time
                    if (next_active_time > 600 and 
                        next_start and last_end and 
                        next_start <= last_end):
                        merge_group.append(next_event)
                        j += 1
                    else:
                        break
                
                # If we have multiple events to merge, create merged PA events
                if len(merge_group) > 1:
                    merged_start, merged_end = self._create_merged_pa_events(merge_group)
                    merged_events.append({
                        'start_event': merged_start,
                        'end_event': merged_end,
                        'indexed_event': merge_group[0]['indexed_event'],  # Store first indexed event for reference
                        'active_time_s': sum(e['active_time_s'] for e in merge_group)
                    })
                    i = j
                else:
                    # Single event, add as-is
                    merged_events.append(current)
                    i += 1
            else:
                # Event doesn't qualify for merging, add as-is
                merged_events.append(current)
                i += 1
        
        return merged_events

    def _create_merged_pa_events(
        self,
        merge_group: List[Dict[str, Any]]
    ) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """Create merged PAStart and PAEnd events from a group of consecutive activities."""
        # Determine merged start time (earliest start)
        start_times = [e['indexed_event'].start_time for e in merge_group if e['indexed_event'].start_time]
        if not start_times:
            raise ValueError("No valid start times in merge group")
        merged_start_time = min(start_times)
        
        # Calculate total duration by summing all durations in the merge group
        durations = [e['indexed_event'].duration_s for e in merge_group if e['indexed_event'].duration_s]
        total_duration_seconds = sum(durations) if durations else 0.0
        
        # End time = start time of first event + sum of all durations
        merged_end_time = merged_start_time + timedelta(seconds=total_duration_seconds)
        
        # Aggregate attributes from all events in the group
        # Use the first event's structure and aggregate numeric values
        first_event = merge_group[0]['start_event']
        merged_attributes = []
        
        # Get all unique attribute names
        attr_names = set()
        for pa_pair in merge_group:
            for attr in pa_pair['start_event']['behaviorEventTypeAttributes']:
                attr_names.add(attr['name'])
        
        # Aggregate attributes
        for attr_name in sorted(attr_names):
            # Special handling for duration_s - use the calculated total duration
            if attr_name == 'duration_s':
                merged_attributes.append({
                    'name': 'duration_s',
                    'value': round_for_storage(total_duration_seconds),
                    'time': merged_end_time.isoformat()
                })
                continue
            
            values = []
            for pa_pair in merge_group:
                for attr in pa_pair['start_event']['behaviorEventTypeAttributes']:
                    if attr['name'] == attr_name:
                        values.append(attr['value'])
                        break
            
            # For numeric attributes, sum them; for others, use first value
            if values:
                if isinstance(values[0], (int, float)):
                    aggregated_value = sum(self._to_float(v) for v in values)
                else:
                    aggregated_value = values[0]
                
                merged_attributes.append({
                    'name': attr_name,
                    'value': round_for_storage(aggregated_value) if isinstance(aggregated_value, float) else aggregated_value,
                    'time': merged_start_time.isoformat()
                })
        
        # Generate event IDs
        start_event_id = self._next_behavior_event_id('PAStart')
        end_event_id = self._next_behavior_event_id('PAEnd')
        
        # Create merged start event
        merged_start_event = {
            'id': start_event_id,
            'behaviorEventType': 'PAStart',
            'time': merged_start_time.isoformat(),
            'behaviorEventTypeAttributes': merged_attributes,
            'sensorEventTypes': first_event.get('sensorEventTypes', []).copy(),
            'relationships': []
        }
        
        # Create merged end event
        end_event_attributes = []
        for attr in merged_attributes:
            end_attr = attr.copy()
            end_attr['time'] = merged_end_time.isoformat()
            end_event_attributes.append(end_attr)
        
        merged_end_event = {
            'id': end_event_id,
            'behaviorEventType': 'PAEnd',
            'time': merged_end_time.isoformat(),
            'behaviorEventTypeAttributes': end_event_attributes,
            'sensorEventTypes': first_event.get('sensorEventTypes', []).copy(),
            'relationships': []
        }
        
        # Link to all original activity events (bidirectional with inverse qualifiers)
        # PA events are inferred from Activity events, so:
        # - DA → PA: 'infer' (DA infers PA from Activity)
        # - PA → Activity: 'inferredBy' (PA is inferred by Activity - Activity is source)
        # - Activity → PA: 'infer' (Activity infers PA - bidirectional)
        participant_id = None
        for pa_pair in merge_group:
            activity_event = pa_pair['indexed_event'].event
            if not participant_id:
                participant_id = pa_pair['indexed_event'].participant_id
            
            if not activity_event.get('relationships'):
                activity_event['relationships'] = []
            # PA → Activity: PA events are inferred by Activity (Activity is the source)
            add_relationship(merged_start_event['relationships'], activity_event['id'], 'inferredBy')
            add_relationship(merged_end_event['relationships'], activity_event['id'], 'inferredBy')
            # Activity → PA: Activity infers PA (bidirectional)
            add_relationship(activity_event['relationships'], start_event_id, 'infer')
            add_relationship(activity_event['relationships'], end_event_id, 'infer')
        
        if participant_id:
            add_relationship(merged_start_event['relationships'], participant_id, 'corr')
            add_relationship(merged_end_event['relationships'], participant_id, 'corr')
        
        return merged_start_event, merged_end_event

    def _next_behavior_event_id(self, event_type: str) -> str:
        self.behavior_event_counter[event_type] += 1
        prefix = {
            'PAStart': 'PAStart',
            'PAEnd': 'PAEnd'
        }.get(event_type, 'PA')
        return f"{prefix}_{self.behavior_event_counter[event_type]}"

    def add_object_type(self) -> None:
        """Add DA object type to OCEL data."""
        register_object_type(
            self.ocel_data,
            'DA',
            [
                {'name': 'calendar_day', 'type': 'string'},
                {'name': 'total_active_kcal', 'type': 'number'},
                {'name': 'total_step_count', 'type': 'number'},
                {'name': 'total_distance_m', 'type': 'number'},
                {'name': 'total_duration_s', 'type': 'number'}
            ],
            verbose=False
        )

    def add_event_types(self) -> None:
        """Add PAStart and PAEnd behavior event types."""
        behavior_event_types = self.ocel_data.get('behaviorEventTypes', [])
        
        # Get attributes from an existing Activity event type to use as template
        activity_event_type = None
        for et in behavior_event_types:
            if et.get('name') == 'Activity':
                activity_event_type = et
                break
        
        # Determine attributes to use
        if activity_event_type:
            attributes = activity_event_type.get('attributes', []).copy()
        else:
            # Fallback: create with common attributes
            attributes = [
                {'name': 'activity_type', 'type': 'string'},
                {'name': 'activity_Kcal', 'type': 'number'},
                {'name': 'step_count', 'type': 'number'},
                {'name': 'distance_m', 'type': 'number'},
                {'name': 'duration_s', 'type': 'number'},
                {'name': 'active_time_s', 'type': 'number'},
                {'name': 'intensity', 'type': 'string'}
            ]
        
        # Register PAStart and PAEnd behavior event types
        for event_type_name in ['PAStart', 'PAEnd']:
            register_behavior_event_type(
                self.ocel_data,
                event_type_name,
                attributes,
                verbose=False
            )

    def infer(self) -> Dict[str, Any]:
        print("\n[DA Inference]")
        self.add_object_type()
        self.add_event_types()
        self.infer_da_objects()
        self.infer_pa_behavior_events()

        enhanced_ocel = self.ocel_data.copy()
        enhanced_ocel = self._merge_inferred_data(enhanced_ocel)

        print(f"  ✓ Created {len(self.inferred_objects)} DA object(s)")
        print(f"  ✓ Created {len(self.inferred_behavior_events)} PA behavior event(s)")
        return enhanced_ocel


class InferencePipeline:
    """Simple pipeline manager for running inference steps in order with dependency management."""
    
    def __init__(self, ocel_data: Dict[str, Any]):
        """
        Initialize pipeline with OCEL data.
        
        Args:
            ocel_data: Dictionary containing Sensor-Augmented OCEL data
        """
        self.ocel_data = ocel_data
        self.inference_steps: List[BaseInference] = []
        self.results: List[Dict[str, Any]] = []
    
    def add_step(self, inference_class: type) -> 'InferencePipeline':
        """
        Add an inference step to the pipeline.
        
        Args:
            inference_class: Inference class to instantiate (PPGRInference, DGVDailyInference, etc.)
            
        Returns:
            Self for method chaining
        """
        # Use the last result as input for the next step, or original data for first step
        input_data = self.results[-1] if self.results else self.ocel_data
        inference = inference_class(input_data)
        self.inference_steps.append(inference)
        enhanced_ocel = inference.infer()
        self.results.append(enhanced_ocel)
        return self
    
    def run(self) -> Dict[str, Any]:
        """
        Run all inference steps in order and return final enhanced OCEL data.
        
        Returns:
            Enhanced OCEL dictionary with all inferred data
        """
        return self.results[-1] if self.results else self.ocel_data
    
    def get_step(self, index: int) -> Optional[BaseInference]:
        """
        Get inference step by index.
        
        Args:
            index: Step index (0-based)
            
        Returns:
            Inference instance or None if index is out of range
        """
        if 0 <= index < len(self.inference_steps):
            return self.inference_steps[index]
        return None



In [2]:
input_path = Path('output/participants_sensor_augmented_ocel-transformed.json')
output_path = Path('output/participants_sensor_augmented_ocel-inferred-cleaning.json')

print("="*80)
print("INFERENCE PIPELINE (PPGR + DGV + DA)")
print("="*80)
print(f"\nLoading OCEL data from {input_path}...")
with open(input_path, 'r', encoding='utf-8') as f:
    ocel_data = json.load(f)

initial_sensor_events = len(ocel_data.get('sensorEvents', []))
initial_behavior_events = len(ocel_data.get('behaviorEvents', []))
initial_objects = len(ocel_data.get('objects', []))

print(f"  ✓ Loaded {initial_sensor_events:,} sensor events")
print(f"  ✓ Loaded {initial_behavior_events:,} behavior events")
print(f"  ✓ Loaded {initial_objects:,} objects")

# Run inference pipeline: PPGR → DGV → DA
print("\n" + "="*80)
print("RUNNING INFERENCE PIPELINE")
print("="*80)
pipeline = InferencePipeline(ocel_data)
pipeline.add_step(PPGRInference)
pipeline.add_step(DGVDailyInference)
pipeline.add_step(DADailyInference)
enhanced_ocel = pipeline.run()

# Get inference instances for summary
ppgr_inference = pipeline.get_step(0)
dgv_inference = pipeline.get_step(1)
da_inference = pipeline.get_step(2)

print(f"\nSaving enhanced OCEL data to {output_path}...")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(enhanced_ocel, f, indent=2, ensure_ascii=False)

print("  ✓ File saved successfully")

# Calculate final counts
final_sensor_events = len(enhanced_ocel.get('sensorEvents', []))
final_behavior_events = len(enhanced_ocel.get('behaviorEvents', []))
final_objects = len(enhanced_ocel.get('objects', []))

print("\n" + "="*80)
print("INFERENCE SUMMARY")
print("="*80)

# Overall statistics
print("\n📊 Overall Statistics:")
print(f"  Sensor Events:     {initial_sensor_events:,} → {final_sensor_events:,} ({final_sensor_events - initial_sensor_events:+,})")
print(f"  Behavior Events:   {initial_behavior_events:,} → {final_behavior_events:,} ({final_behavior_events - initial_behavior_events:+,})")
print(f"  Objects:           {initial_objects:,} → {final_objects:,} ({final_objects - initial_objects:+,})")

# PPGR Inference Results
if ppgr_inference:
    ppbg_object_count = sum(1 for obj in ppgr_inference.inferred_objects if obj.get('type') == 'PPBG')
    ppic_object_count = sum(1 for obj in ppgr_inference.inferred_objects if obj.get('type') == 'PPIC')
    peak_count = sum(1 for e in ppgr_inference.inferred_behavior_events if e.get('behaviorEventType') == 'PPBGPeak')
    recovery_count = sum(1 for e in ppgr_inference.inferred_behavior_events if e.get('behaviorEventType') == 'PPBGRecovery')
    
    print("\n🍽️  Postprandial Glucose Response (PPGR):")
    print(f"  Objects:")
    print(f"    • PPBG: {ppbg_object_count:,}")
    print(f"    • PPIC: {ppic_object_count:,}")
    print(f"  Behavior Events:")
    print(f"    • PPBGPeak: {peak_count:,}")
    print(f"    • PPBGRecovery: {recovery_count:,}")
    print(f"    • Total: {len(ppgr_inference.inferred_behavior_events):,}")

# DGV Inference Results
if dgv_inference:
    hypoglycemia_events = [e for e in dgv_inference.inferred_behavior_events if e['behaviorEventType'].startswith('Hypoglycemia')]
    hyperglycemia_events = [e for e in dgv_inference.inferred_behavior_events if e['behaviorEventType'].startswith('Hyperglycemia')]
    hypo_start = sum(1 for e in hypoglycemia_events if e['behaviorEventType'] == 'HypoglycemiaStart')
    hypo_end = sum(1 for e in hypoglycemia_events if e['behaviorEventType'] == 'HypoglycemiaEnd')
    hyper_start = sum(1 for e in hyperglycemia_events if e['behaviorEventType'] == 'HyperglycemiaStart')
    hyper_end = sum(1 for e in hyperglycemia_events if e['behaviorEventType'] == 'HyperglycemiaEnd')
    
    print("\n📈 Daily Glycemic Variability (DGV):")
    print(f"  Objects: {len(dgv_inference.inferred_objects):,}")
    print(f"  Behavior Events:")
    print(f"    • Hypoglycemia: {len(hypoglycemia_events):,} (Start: {hypo_start:,}, End: {hypo_end:,})")
    print(f"    • Hyperglycemia: {len(hyperglycemia_events):,} (Start: {hyper_start:,}, End: {hyper_end:,})")
    print(f"    • Total: {len(dgv_inference.inferred_behavior_events):,}")

# DA Inference Results
if da_inference:
    pa_start_count = sum(1 for e in da_inference.inferred_behavior_events if e.get('behaviorEventType') == 'PAStart')
    pa_end_count = sum(1 for e in da_inference.inferred_behavior_events if e.get('behaviorEventType') == 'PAEnd')
    
    print("\n🏃 Daily Activity (DA):")
    print(f"  Objects: {len(da_inference.inferred_objects):,}")
    print(f"  Behavior Events:")
    print(f"    • PAStart: {pa_start_count:,}")
    print(f"    • PAEnd: {pa_end_count:,}")
    print(f"    • Total: {len(da_inference.inferred_behavior_events):,}")

# Per-participant statistics
if da_inference and dgv_inference:
    participant_set = set()
    object_map = {}
    for obj in enhanced_ocel.get('objects', []):
        object_map[obj['id']] = obj
        if obj.get('type') == 'Participant':
            participant_set.add(obj['id'])
    
    da_by_participant = defaultdict(int)
    for obj in da_inference.inferred_objects:
        if obj.get('type') == 'DA':
            for rel in obj.get('relationships', []):
                rel_target = rel.get('objectId') or rel.get('id')
                if rel_target and rel.get('qualifier') == 'corr':
                    # Check if it's a participant using object_map
                    target_obj = object_map.get(rel_target)
                    if target_obj and target_obj.get('type') == 'Participant':
                        da_by_participant[rel_target] += 1
                        break
    
    dgv_by_participant = defaultdict(int)
    for obj in dgv_inference.inferred_objects:
        if obj.get('type') == 'DGV':
            for rel in obj.get('relationships', []):
                rel_target = rel.get('objectId') or rel.get('id')
                if rel_target and rel.get('qualifier') == 'corr':
                    target_obj = object_map.get(rel_target)
                    if target_obj and target_obj.get('type') == 'Participant':
                        dgv_by_participant[rel_target] += 1
                        break
    
    print("\n👥 Per-Participant Object Counts:")
    print("  " + "-" * 76)
    for participant_id in sorted(participant_set):
        da_count = da_by_participant.get(participant_id, 0)
        dgv_count = dgv_by_participant.get(participant_id, 0)
        diff = da_count - dgv_count
        diff_str = f" (+{diff})" if diff > 0 else f" ({diff})" if diff < 0 else " (equal)"
        print(f"    {participant_id}: DA={da_count:,}, DGV={dgv_count:,}{diff_str}")

print("\n" + "="*80)



INFERENCE PIPELINE (PPGR + DGV + DA)

Loading OCEL data from output/participants_sensor_augmented_ocel-transformed.json...
  ✓ Loaded 677,991 sensor events
  ✓ Loaded 260,589 behavior events
  ✓ Loaded 19 objects

RUNNING INFERENCE PIPELINE

[PPGR Inference]
Starting postprandial object inference (PPBG + PPIC)...
  ✓ Processed 4009 meals (337 skipped due to insufficient data)
  ✓ Created 4009 PPBG object(s), 2246 PPIC object(s)

[DGV Inference]
Starting DGV object inference...
  ✓ Created 1678 DGV object(s)
  ✓ Linked 1257 SleepTime event(s) to 1678 DGV object(s)
Starting DGV behavior event inference...
  ✓ Created 21381 DGV behavior event(s)

[DA Inference]
Inferring PA behavior events from Activity events...
  ✓ Created 68790 PA behavior event(s)
  ✓ Created 1956 DA object(s)
  ✓ Created 68790 PA behavior event(s)

Saving enhanced OCEL data to output/participants_sensor_augmented_ocel-inferred-cleaning.json...
  ✓ File saved successfully

INFERENCE SUMMARY

📊 Overall Statistics:
  Se